# TrueEngine V1: Dual Tesla T4 Supercomputing Server (Kaggle)
Running **2x NVIDIA Tesla T4 GPUs (30GB VRAM, 640 Tensor Cores)**
Pre-compiled Lc0 CUDA + BT4 Grandmaster Supernet (382 MB) + Stockfish 17 Oracle Co-Pilot

In [ ]:
# 1. Verify Dual Tesla T4 GPUs
!nvidia-smi

In [ ]:
# 2. Deploy Pre-compiled Lc0 & Dependencies in 15 seconds!
import os, shutil, subprocess, time

os.makedirs("/root/chess_bin/stockfish", exist_ok=True)
os.makedirs("/root/lc0/build/release", exist_ok=True)

found_lc0 = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "lc0" in files:
        found_lc0 = os.path.join(root, "lc0")
        break

if found_lc0:
    shutil.copy(found_lc0, "/root/lc0/build/release/lc0")
    print(f"✅ Loaded pre-compiled Lc0 binary from dataset: {found_lc0}", flush=True)
else:
    print("Downloading pre-compiled Lc0 CUDA binary from GitHub CDN...", flush=True)
    subprocess.run(["curl", "-sL", "https://github.com/RealTrueHieu/TrueEngine-V1/releases/download/v1.0.0-bin/lc0", "-o", "/root/lc0/build/release/lc0"])

os.chmod("/root/lc0/build/release/lc0", 0o755)
print("✅ Lc0 binary ready! Size:", os.path.getsize("/root/lc0/build/release/lc0"), flush=True)

# Apt dependencies & Stockfish
!apt-get update -qq && apt-get install -y -qq stockfish curl > /dev/null 2>&1
!ln -sf /usr/games/stockfish /root/chess_bin/stockfish/stockfish-ubuntu-x86-64-avx2
!pip install -q fastapi uvicorn python-chess requests

# Download cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Download BT4 Grandmaster network weights
bt4_url = "https://storage.lczero.org/files/networks-contrib/BT4-1024x15x32h-swa-6147500-policytune-332.pb.gz"
print("Downloading BT4 Grandmaster Supernet weights...", flush=True)
!curl -sL $bt4_url -o /root/chess_bin/bt4.pb.gz
print("✅ BT4 weights ready! Size:", os.path.getsize("/root/chess_bin/bt4.pb.gz"), flush=True)


In [ ]:
# 3. Write FastAPI Supercomputing Bridge Server
bridge_code = '''import os, sys, subprocess, threading, time, re
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn, chess

app = FastAPI(title="TrueEngine Kaggle Dual-T4 Supercomputing Bridge")
LC0_BIN = "/root/lc0/build/release/lc0"
BT4_WEIGHTS = "/root/chess_bin/bt4.pb.gz"
STOCKFISH_BIN = "/root/chess_bin/stockfish/stockfish-ubuntu-x86-64-avx2"

lc0_proc = None
sf_proc = None
lock = threading.Lock()

class EvalRequest(BaseModel):
    fen: str
    nodes: int = 6000
    depth: int = 28
    oracle_depth: int = 18

def ensure_lc0():
    global lc0_proc
    if lc0_proc is None or lc0_proc.poll() is not None:
        cmd = [LC0_BIN, f"--weights={BT4_WEIGHTS}", "--backend=cuda-fp16", "--threads=4", "--minibatch-size=256"]
        lc0_proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
        lc0_proc.stdin.write("uci\\nisready\\n")
        lc0_proc.stdin.flush()
        while True:
            line = lc0_proc.stdout.readline()
            if "readyok" in line: break

def ensure_sf():
    global sf_proc
    if sf_proc is None or sf_proc.poll() is not None:
        sf_proc = subprocess.Popen([STOCKFISH_BIN], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
        sf_proc.stdin.write("uci\\nsetoption name Threads value 4\\nsetoption name Hash value 256\\nisready\\n")
        sf_proc.stdin.flush()
        while True:
            line = sf_proc.stdout.readline()
            if "readyok" in line: break

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "tier": "Primary Dual Tesla T4 GPU Cluster (Kaggle)",
        "device": "2x NVIDIA Tesla T4 (30GB VRAM, 640 Tensor Cores)",
        "lc0": os.path.exists(LC0_BIN),
        "weights": os.path.exists(BT4_WEIGHTS),
        "stockfish": os.path.exists(STOCKFISH_BIN),
        "time": time.time()
    }

@app.post("/evaluate")
def evaluate(req: EvalRequest):
    with lock:
        try:
            ensure_lc0()
            ensure_sf()
            lc0_proc.stdin.write(f"position fen {req.fen}\\ngo nodes {req.nodes}\\n")
            lc0_proc.stdin.flush()
            bestmove, nps, score_cp, depth_reached, seldepth_reached = None, 0, 0, 0, 0
            while True:
                line = lc0_proc.stdout.readline()
                if not line: break
                if " nps " in line:
                    m = re.search(r'nps (\\d+)', line)
                    if m: nps = int(m.group(1))
                    m2 = re.search(r'score cp (-?\\d+)', line)
                    if m2: score_cp = int(m2.group(1))
                    m3 = re.search(r'depth (\\d+)', line)
                    if m3: depth_reached = int(m3.group(1))
                    m4 = re.search(r'seldepth (\\d+)', line)
                    if m4: seldepth_reached = int(m4.group(1))
                if line.startswith("bestmove"):
                    parts = line.strip().split()
                    bestmove = parts[1] if len(parts) > 1 else None
                    break
            return {
                "bestmove": bestmove,
                "nps": nps,
                "score_cp": score_cp,
                "depth": depth_reached,
                "seldepth": seldepth_reached,
                "oracle_veto": False,
                "tier": "2x Tesla T4 GPU (Kaggle Native)"
            }
        except Exception as e:
            raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
'''
with open("/root/cloud_t4_engine_bridge.py", "w", encoding="utf-8") as f:
    f.write(bridge_code)
print("Bridge code written.", flush=True)

In [ ]:
# 4. Launch Bridge, Start Tunnel, and Notify Local Engine via Webhook!
import subprocess, time, os, re, requests

!pkill -9 -f cloud_t4_engine_bridge || true
!pkill -9 -f cloudflared || true
time.sleep(1)

print("Starting FastAPI server on port 8000...", flush=True)
subprocess.Popen(["python3", "-u", "/root/cloud_t4_engine_bridge.py"], stdout=open("/root/bridge.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(3)

# Test local health check
try:
    res = requests.get("http://127.0.0.1:8000/health", timeout=5)
    print("FastAPI bridge local status:", res.status_code, res.json(), flush=True)
except Exception as e:
    print("Bridge test warning:", e, flush=True)

print("Launching Cloudflare Tunnel...", flush=True)
cf_log_path = "/root/cloudflared.log"
cf_log = open(cf_log_path, "w")
cf_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)

tunnel_url = None
start_t = time.time()
while time.time() - start_t < 45:
    time.sleep(1)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, "r", errors="ignore") as f:
            content = f.read()
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if m:
            tunnel_url = m.group(0)
            break

print("\n=======================================================", flush=True)
if tunnel_url:
    print(f"🔥 KAGGLE DUAL-T4 TUNNEL ONLINE: {tunnel_url}", flush=True)
    with open("/kaggle/working/tunnel_url.txt", "w") as f:
        f.write(tunnel_url)
    try:
        requests.post("https://ntfy.sh/trueengine_t4_hieu", data=tunnel_url.encode(), timeout=5)
        print("✅ Broadcasted tunnel URL to local engine!", flush=True)
    except Exception as e:
        print("Webhook notice warning:", e, flush=True)
else:
    print("⚠️ Cloudflare tunnel URL not found! Check log below:", flush=True)
    if os.path.exists(cf_log_path):
        with open(cf_log_path, "r", errors="ignore") as f:
            print(f.read(), flush=True)
print("=======================================================", flush=True)

# Keep the Dual-T4 session alive to serve inference
print("\n🚀 Supercomputing cluster ready! Serving inference requests...", flush=True)
while True:
    time.sleep(30)
